# 🕵️ Dia 67 — Fraude de recrutamento: os mecanismos e um checklist pontuado por evidência

**Investigador Cyber 90 Dias** · Fase 5 · Rank Jutsu `A`

> Baseado no *Day 67* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-5-fraud-social-engineering/day-67.md) (*Recruitment fraud, the mechanisms and an evidence-scored checklist*), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Quarta, 21 de outubro. A dona da conta **L-03** registrou um relato: achava que tinha um emprego. Chegaram mais três relatos anonimizados de falsas vagas (E1 a E4).

Rafa: *"Mecanismo presente só com citação. Relato calado é 'informação insuficiente'."*

Neste notebook você vai:

1. Transformar a árvore das **três respostas** numa função que recusa "presente" sem citação literal.
2. Pontuar o relato **E2** célula por célula, num quiz guiado.
3. Passar o scorecard por um **validador** (citação inexistente, coluna faltando, "perfeito demais").
4. Extrair **indicadores** com expressões regulares e gravar o CSV para o Dia 68.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Todos os dados são **fictícios** (domínios `.example`, telefones `+55 00 00000-00NN`, chaves Pix `PIXFICT-*`).

## 1. Os relatos (fictícios) e o cabeçalho do original

Embutidos no notebook: funcionam offline. São os mesmos de `caso-continuo/dados/dia-67/`.

In [ ]:
RELATOS_TXT = "[E1] Pessoa recrutada (dona da conta L-03) respondeu a anuncio de 'Conferente de pagamentos home office' da 'Agil Recebiveis'. No mesmo dia, o recrutador escreveu do WhatsApp +55 00 00000-0011 pedindo para continuar por la. A 'entrevista' foi por mensagem de texto. Pediram 'foto do documento e selfie para o cadastro' antes de qualquer contrato. A tarefa: 'receber transferencias de clientes na sua conta e repassar para a chave da empresa, ficando com 5%'. Chave de recebimento: PIXFICT-L03 (ficticia), a propria conta da recrutada.\n\n[E2] Vitima de outra cooperativa (coopvale.example) aceitou 'trabalho de avaliar produtos e curtir videos' da 'Agil Recebiveis', contato pelo +55 00 00000-0012. O app agil-tarefas.example mostrava saldo de R$ 640, mas o saque exigia uma 'taxa de liberacao' de R$ 150 por Pix para PIXFICT-L02 (ficticia). Depois, novo nivel pedia R$ 900. O relato nao menciona pedido de documentos.\n\n[E3] Candidata a 'auxiliar de estoque' da 'Vagas Serra Logistica' foi aprovada sem entrevista e precisou pagar R$ 89 de 'uniforme e exame admissional' por Pix para PIXFICT-X9 (ficticia). Contato: +55 00 00000-0013. A mensagem dizia que a 'taxa de liberacao' seria devolvida no primeiro salario. Nao houve pedido de reenvio de dinheiro nem de pacotes.\n\n[E4] Anuncio da 'Agil Recebiveis' para 'atendimento ao cliente no exterior, salario em dolar, passagem e visto por nossa conta'. O recrutador (+55 00 00000-0011) disse para 'viajar com visto de turista, depois a empresa regulariza' e nao informou a cidade de destino. A pessoa desistiu e registrou o relato.\n"

import re
# Separa o arquivo em {código: texto}. Cada relato começa com [E1], [E2]...
RELATOS = dict(re.findall(r"\[(E\d)\]\s*(.+?)(?=\n\n\[E\d\]|\Z)", RELATOS_TXT, flags=re.S))

# Cabeçalho idêntico ao do Dia 67 original (o Dia 68 importa por estes nomes).
CABECALHO = ("case_id,source_link,report_date,first_contact_channel,interview,upfront_fee,fake_check,task_scam,"
             "reshipping,money_mule,identity_harvest,forced_labor_recruitment,off_platform_move,phones,emails,"
             "domains,payment_handles,crypto_addresses,company_names,app_names,phrases").split(",")
MECANISMOS = ["upfront_fee", "fake_check", "task_scam", "reshipping", "money_mule",
              "identity_harvest", "forced_labor_recruitment", "off_platform_move"]

for codigo, texto in RELATOS.items():
    print(f"{codigo}: {texto[:110]}...")

## 2. A árvore das três respostas, em código

A função `pontuar()` recebe o relato, a sua resposta e (se for "presente") a citação. Ela **devolve "informação insuficiente"** quando a citação não está literalmente no relato: é o ramo "palpite seu" do fluxograma.

In [ ]:
import unicodedata

def normal(texto):
    """Minúsculas, sem acento e com espaços únicos: compara o conteúdo, não a digitação."""
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return " ".join(texto.lower().split())

def pontuar(relato, resposta, citacao=""):
    """
    resposta: 'presente', 'ausente' ou 'informação insuficiente'.
    Devolve o valor que vai para a célula do scorecard.
    """
    if resposta == "presente":
        # Q1 + CHK do fluxograma: a citação tem de ser palavras da fonte.
        if citacao and normal(citacao) in normal(relato):
            return f'presente: "{citacao}"'
        return "informação insuficiente"       # palpite seu → rebaixa
    if resposta == "ausente":
        return "ausente"                        # só quando o relato diz que NÃO aconteceu
    return "informação insuficiente"            # relato omisso

# Duas tentativas no relato E2:
print(pontuar(RELATOS["E2"], "presente", "o saque exigia uma 'taxa de liberacao' de R$ 150"))
print(pontuar(RELATOS["E2"], "presente", "pediram o RG depois, como sempre"))   # paráfrase/palpite

## 3. 🎮 Pontue o relato E2

Para cada mecanismo, escolha **a** (presente), **b** (ausente) ou **c** (informação insuficiente). Leia o relato acima antes. `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
MODO_JOGO = True

# Gabarito do E2: (mecanismo, letra certa, por quê)
GABARITO_E2 = [
    ("upfront_fee", "b", "A taxa vem no SAQUE, não antes de começar: é golpe de tarefas, não taxa antecipada clássica."),
    ("fake_check", "c", "O relato não fala de cheque nem de transferência com excedente."),
    ("task_scam", "a", "\"o saque exigia uma 'taxa de liberacao' de R$ 150\" e depois \"novo nivel pedia R$ 900\"."),
    ("reshipping", "b", "Nenhuma menção a pacotes; o relato descreve só tarefas no app."),
    ("money_mule", "b", "A vítima pagou; não recebeu nem repassou dinheiro de terceiros."),
    ("identity_harvest", "c", "\"O relato nao menciona pedido de documentos.\" Silêncio ≠ ausente."),
    ("forced_labor_recruitment", "b", "Trabalho remoto local, sem viagem."),
    ("off_platform_move", "c", "O primeiro contato já foi por mensagem; o relato não diz que saiu de outra plataforma."),
]
LETRAS = {"a": "presente", "b": "ausente", "c": "informação insuficiente"}

acertos = 0
for mecanismo, certa, porque in GABARITO_E2:
    print(f"\n▶ {mecanismo}")
    if MODO_JOGO:
        r = ""
        while r not in LETRAS:
            r = input("   a presente / b ausente / c informação insuficiente: ").strip().lower()
        acertos += (r == certa)
        print("   ✅" if r == certa else "   ❌")
    print(f"   Gabarito: {LETRAS[certa]} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(GABARITO_E2)}")

> 💡 Discutível de propósito: `upfront_fee` em E2. Alguém pode argumentar "presente", porque houve pagamento antes de qualquer salário (sinal 1 do checklist). O original resolve assim no exemplo resolvido: *"Ausente na forma clássica"*, e registra o mecanismo certo (`task_scam`). Se você discordar, escreva o argumento nas notas, com citação. É assim que se discute um scorecard.

## 4. O validador do scorecard

Três regras, todas tiradas do checkpoint do original:

1. toda célula "presente" tem citação entre aspas, **e a citação existe no relato**;
2. o cabeçalho tem todas as colunas que o Dia 68 importa;
3. um scorecard **sem nenhuma lacuna** recebe aviso: relatos publicados costumam ser incompletos.

Abaixo ele roda no gabarito `scorecard-br.csv` (embutido). Para validar o seu, suba o arquivo no Colab e mude `ARQUIVO_DO_ALUNO`.

In [ ]:
SCORECARD_BR = 'case_id,source_link,report_date,first_contact_channel,interview,upfront_fee,fake_check,task_scam,reshipping,money_mule,identity_harvest,forced_labor_recruitment,off_platform_move,phones,emails,domains,payment_handles,crypto_addresses,company_names,app_names,phrases\nE1,relatos-recrutamento.txt#E1,2026-10-21,anúncio de emprego,texto apenas,ausente,informação insuficiente,ausente,ausente,"presente: ""receber transferencias de clientes na sua conta e repassar para a chave da empresa, ficando com 5%""","presente: ""foto do documento e selfie para o cadastro"" antes de qualquer contrato",ausente,"presente: ""pedindo para continuar por la"" no mesmo dia",+55 00 00000-0011,,,PIXFICT-L03,,Agil Recebiveis,,conferente de pagamentos\nE2,relatos-recrutamento.txt#E2,2026-10-21,mensagem não solicitada,informação insuficiente,ausente,informação insuficiente,"presente: ""o saque exigia uma \'taxa de liberacao\' de R$ 150""; depois ""novo nivel pedia R$ 900""",ausente,ausente,informação insuficiente,ausente,informação insuficiente,+55 00 00000-0012,,agil-tarefas.example,PIXFICT-L02,,Agil Recebiveis,agil-tarefas,taxa de liberacao\nE3,relatos-recrutamento.txt#E3,2026-10-22,anúncio de emprego,"nenhuma (""aprovada sem entrevista"")","presente: ""pagar R$ 89 de \'uniforme e exame admissional\'""",informação insuficiente,ausente,ausente,ausente,informação insuficiente,ausente,informação insuficiente,+55 00 00000-0013,,,PIXFICT-X9,,Vagas Serra Logistica,,taxa de liberacao\nE4,relatos-recrutamento.txt#E4,2026-10-22,anúncio de emprego,informação insuficiente,informação insuficiente,informação insuficiente,informação insuficiente,informação insuficiente,informação insuficiente,informação insuficiente,"presente: ""passagem e visto por nossa conta""; ""viajar com visto de turista""; destino não informado",informação insuficiente,+55 00 00000-0011,,,,,Agil Recebiveis,,\n'

import csv, io
from pathlib import Path

ARQUIVO_DO_ALUNO = "day67-scorecard.csv"   # se existir aqui, ele é validado no lugar do gabarito

def validar(texto_csv, relatos):
    leitor = csv.DictReader(io.StringIO(texto_csv))
    problemas, lacunas, total = [], 0, 0
    faltando = [c for c in CABECALHO if c not in (leitor.fieldnames or [])]
    if faltando:
        problemas.append(f"Colunas faltando: {faltando}")
    linhas = list(leitor)
    for linha in linhas:
        caso = linha["case_id"]
        fonte = relatos.get(caso, "")
        for mec in MECANISMOS:
            valor = (linha.get(mec) or "").strip()
            total += 1
            if valor.startswith("informação insuficiente") or valor.startswith("not enough"):
                lacunas += 1
            elif valor.startswith(("presente", "present")):
                citacoes = re.findall(r'"([^"]+)"', valor)
                if not citacoes:
                    problemas.append(f"{caso}/{mec}: 'presente' sem citação entre aspas")
                for cit in citacoes:
                    if fonte and normal(cit) not in normal(fonte):
                        problemas.append(f"{caso}/{mec}: citação não está no relato → {cit!r}")
    return linhas, problemas, lacunas, total

texto = Path(ARQUIVO_DO_ALUNO).read_text(encoding="utf-8") if Path(ARQUIVO_DO_ALUNO).is_file() else SCORECARD_BR
origem = ARQUIVO_DO_ALUNO if texto is not SCORECARD_BR else "gabarito scorecard-br.csv"
linhas, problemas, lacunas, total = validar(texto, RELATOS)

print(f"Validando: {origem} ({len(linhas)} casos)")
print("\n".join("❌ " + p for p in problemas) or "✅ Todas as citações existem nos relatos.")
print(f"Lacunas ('informação insuficiente'): {lacunas} de {total} células de mecanismo")
if lacunas == 0:
    print("⚠️  Nenhuma lacuna. Relatos reais quase nunca dizem tudo: alguém preencheu por palpite?")

## 5. Extrair indicadores (o que o Dia 68 vai ligar)

Regex simples para os formatos fictícios do caso. Repare que a extração **não interpreta**: copia o valor exato. "Telefone parecido" não é indicador.

In [ ]:
PADROES = {
    "phones": r"\+55 00 00000-\d{4}",            # formato fictício do repositório
    "domains": r"\b[\w-]+\.example\b",           # só .example (RFC 2606)
    "payment_handles": r"\bPIXFICT-[A-Z0-9]+\b", # chaves Pix fictícias
}
EMPRESAS = ["Agil Recebiveis", "Vagas Serra Logistica"]   # nomes que os relatos citam entre aspas

extraidos = {}
for caso, texto in RELATOS.items():
    achados = {col: sorted(set(re.findall(p, texto))) for col, p in PADROES.items()}
    achados["company_names"] = [e for e in EMPRESAS if e.lower() in texto.lower()]
    # A cooperativa DA VÍTIMA (coopvale.example) aparece no texto, mas não é indicador do golpe:
    # é contexto. Extração automática exige revisão humana antes de virar nó.
    achados["domains"] = [d for d in achados["domains"] if d != "coopvale.example"]
    extraidos[caso] = achados
    print(caso, {k: v for k, v in achados.items() if v})

# Contagem que interessa amanhã: valores que aparecem em MAIS de um relato
from collections import Counter
contagem = Counter(v for a in extraidos.values() for vals in a.values() for v in vals)
print("\nValores em 2+ relatos (candidatos a nó compartilhado no Dia 68):")
for valor, n in contagem.items():
    if n > 1:
        print(f"  {valor}: {n} relatos")

## 6. 📝 Gravar o CSV para o Dia 68 e a contagem de lacunas

In [ ]:
#@title Notas do dia { display-mode: "form" }
minha_contagem_lacunas_casos_reais = "0"  #@param {type:"string"}
argumento_upfront_fee_E2 = "Concordo com 'ausente na forma clássica': a taxa só aparece para sacar."  #@param {type:"string"}

from pathlib import Path
# O CSV que o Dia 68 importa: o gabarito BR (troque pelo seu quando tiver os casos reais).
if not Path("day67-scorecard.csv").is_file():          # não sobrescreve o seu, se você já subiu um
    Path("day67-scorecard.csv").write_text(texto, encoding="utf-8")
Path("notas").mkdir(exist_ok=True)
notas = Path("notas/dia67-contagem-lacunas.md")
notas.write_text(
    "# Dia 67 — contagem de lacunas (fica FORA do CSV, como manda o original)\n\n"
    f"- Lacunas no scorecard BR (E1–E4): {lacunas} de {total}\n"
    f"- Lacunas nos meus três casos reais: {minha_contagem_lacunas_casos_reais}\n"
    f"- Problemas do validador: {len(problemas)}\n"
    f"- upfront_fee em E2: {argumento_upfront_fee_E2}\n"
    "- E4 encaminhado ao fluxo do Dia 70 (indicador de tráfico); nenhuma pesquisa sobre o recrutador.\n",
    encoding="utf-8")
print(notas.read_text(encoding="utf-8"))
print("💾 day67-scorecard.csv e notas/dia67-contagem-lacunas.md prontos")
try:
    from google.colab import files
    files.download("day67-scorecard.csv"); files.download(str(notas))
except ImportError:
    print("(Fora do Colab: arquivos na pasta atual)")

### 🧠 Camada 8

Pé na porta (Freedman e Fraser), compromisso e coerência (Cialdini), aversão à perda (Kahneman e Tversky): o golpe de tarefas sobe a escada com a vítima. No relatório, descreva o **mecanismo**, nunca "a vítima foi gananciosa". E repare na sua própria escada: cada célula vazia pede para ser preenchida.

## ✅ Antes de fechar o Dia 67

- [ ] E1 a E4 pontuados por você antes de olhar o gabarito
- [ ] Validador sem nenhum ❌ no seu scorecard
- [ ] Contagem de lacunas registrada nas **notas**, não no CSV
- [ ] E4 encaminhado ao fluxo do Dia 70, sem pesquisa extra

**Amanhã (Dia 68):** as chaves Pix dos laranjas, os telefones dos recrutadores e o domínio da Gráfica Jatobá viram um grafo. Quem é ponte entre o golpe do Pix bloqueado e o falso emprego?

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>